In [8]:
import pandas as pd
import numpy as np
import requests
import time

print("Imports OK")
print(f"Pandas: {pd.__version__}")
print(f"Numpy : {np.__version__}")


Imports OK
Pandas: 3.0.3
Numpy : 2.4.6


In [9]:
API_KEY = "746a30f11c7a4614b7878680ed13a7e8"

def fetch_batch(symbol="XAU/USD", interval="5min",
                outputsize=5000, end_date=None):
    url    = "https://api.twelvedata.com/time_series"
    params = {
        "symbol"    : symbol,
        "interval"  : interval,
        "outputsize": outputsize,
        "apikey"    : API_KEY,
        "format"    : "JSON"
    }
    if end_date:
        params["end_date"] = end_date
    r    = requests.get(url, params=params)
    data = r.json()
    if data.get("status") == "error":
        print(f"Error: {data.get('message')}")
        return None
    df = pd.DataFrame(data["values"])
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df.set_index("datetime").sort_index()
    df = df.astype(float)
    df.columns = [c.lower() for c in df.columns]
    return df

# Fetch one batch first to test
print("Fetching batch 1...")
batch1 = fetch_batch("XAU/USD", "5min", 5000)

if batch1 is not None:
    print(f"OK: {len(batch1)} bars")
    print(f"Range: {batch1.index[0].date()} "
          f"to {batch1.index[-1].date()}")
else:
    print("Failed")

Fetching batch 1...
OK: 5000 bars
Range: 2026-07-01 to 2026-07-19


In [10]:
print("Fetching all batches...")
all_batches = [batch1]
end_date    = batch1.index[0].strftime("%Y-%m-%d %H:%M:%S")

for i in range(2, 9):
    time.sleep(8)  # respect 8 calls/minute limit
    batch = fetch_batch("XAU/USD", "5min",
                        5000, end_date)
    if batch is None or len(batch) == 0:
        print(f"Batch {i}: failed")
        break
    print(f"Batch {i}: {len(batch)} bars "
          f"| {batch.index[0].date()} "
          f"to {batch.index[-1].date()}")
    all_batches.append(batch)
    end_date = batch.index[0].strftime(
               "%Y-%m-%d %H:%M:%S")

# Combine
df = pd.concat(all_batches).sort_index()
df = df[~df.index.duplicated(keep='last')]

print(f"\nCombined:")
print(f"  Bars  : {len(df)}")
print(f"  Range : {df.index[0].date()} "
      f"to {df.index[-1].date()}")

Fetching all batches...
Batch 2: 5000 bars | 2026-06-14 to 2026-07-01
Batch 3: 5000 bars | 2026-05-27 to 2026-06-14
Batch 4: 5000 bars | 2026-05-10 to 2026-05-27
Batch 5: 5000 bars | 2026-04-23 to 2026-05-10
Batch 6: 5000 bars | 2026-04-05 to 2026-04-23
Batch 7: 5000 bars | 2026-03-19 to 2026-04-05
Batch 8: 5000 bars | 2026-03-02 to 2026-03-19

Combined:
  Bars  : 39993
  Range : 2026-03-02 to 2026-07-19


In [13]:
from datetime import datetime

# Settings
EMA_FAST = 8
EMA_MID  = 21
EMA_SLOW = 89
RSI_LEN  = 14
ATR_LEN  = 14
ADX_LEN  = 14
ADX_MIN  = 15
UTC_OFFSET = 3

def get_anchor_hour_utc(dt):
    year = dt.year
    march_sundays = [d for d in range(1, 32)
                     if datetime(year, 3, d).weekday() == 6]
    dst_start = datetime(year, 3, march_sundays[1])
    nov_sundays = [d for d in range(1, 8)
                   if datetime(year, 11, d).weekday() == 6]
    dst_end = datetime(year, 11, nov_sundays[0])
    if dst_start <= dt.replace(tzinfo=None) < dst_end:
        return 14
    else:
        return 15

def get_cutoff_hour_utc(dt):
    return get_anchor_hour_utc(dt) + 2

# Indicator functions
def ema(series, length):
    return series.ewm(span=length, adjust=False).mean()

def rsi_calc(series, length=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(length).mean()
    loss  = (-delta.clip(upper=0)).rolling(length).mean()
    rs    = gain / loss
    return 100 - (100 / (1 + rs))

def atr_calc(high, low, close, length=14):
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low  - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    return tr.rolling(length).mean()

def adx_calc(high, low, close, length=14):
    tr   = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low  - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    atr_ = tr.rolling(length).mean()
    up   = high.diff()
    down = -low.diff()
    dm_p = pd.Series(np.where(
        (up>down)&(up>0), up, 0), index=high.index)
    dm_m = pd.Series(np.where(
        (down>up)&(down>0), down, 0), index=high.index)
    di_p = 100 * dm_p.rolling(length).mean() / atr_
    di_m = 100 * dm_m.rolling(length).mean() / atr_
    dx   = 100 * (di_p - di_m).abs() / (di_p + di_m)
    return dx.rolling(length).mean()

def macd_calc(series, fast=8, slow=17, signal=9):
    ml = ema(series, fast) - ema(series, slow)
    ms = ml.ewm(span=signal, adjust=False).mean()
    return ml, ms

# Calculate LIP-SIM indicators
print("Calculating indicators...")
df['ema_fast'] = ema(df['close'], EMA_FAST)
df['ema_mid']  = ema(df['close'], EMA_MID)
df['ema_slow'] = ema(df['close'], EMA_SLOW)
df['rsi']      = rsi_calc(df['close'], RSI_LEN)
df['atr']      = atr_calc(df['high'], df['low'],
                           df['close'], ATR_LEN)
df['adx']      = adx_calc(df['high'], df['low'],
                           df['close'], ADX_LEN)
df['macd_l'], df['macd_s'] = macd_calc(df['close'])

# LIP-SIM conditions
df['bull_trend'] = (df['ema_fast'] > df['ema_mid']) & \
                   (df['ema_fast'] > df['ema_slow'])
df['bear_trend'] = (df['ema_fast'] < df['ema_mid']) & \
                   (df['ema_fast'] < df['ema_slow'])
df['rsi_bull']   = (df['rsi'] > 50) & (df['rsi'] < 80)
df['rsi_bear']   = (df['rsi'] < 50) & (df['rsi'] > 20)
df['macd_bull']  = df['macd_l'] > df['macd_s']
df['macd_bear']  = df['macd_l'] < df['macd_s']
df['has_trend']  = df['adx'] >= ADX_MIN

print(f"LIP-SIM done — {len(df)} bars")
print(f"  Bear trend: {df['bear_trend'].sum()} bars")
print(f"  Bull trend: {df['bull_trend'].sum()} bars")

Calculating indicators...
LIP-SIM done — 39993 bars
  Bear trend: 16727 bars
  Bull trend: 14011 bars


In [12]:
print("Calculating SMC components...")
n        = len(df)
lookback = 5

# Swing highs and lows
swing_high = pd.Series(False, index=df.index)
swing_low  = pd.Series(False, index=df.index)
sh_price   = pd.Series(np.nan, index=df.index)
sl_price   = pd.Series(np.nan, index=df.index)

for i in range(lookback, n-lookback):
    wh = df['high'].iloc[i-lookback:i+lookback+1]
    wl = df['low'].iloc[i-lookback:i+lookback+1]
    if df['high'].iloc[i] == wh.max():
        swing_high.iloc[i] = True
        sh_price.iloc[i]   = df['high'].iloc[i]
    if df['low'].iloc[i] == wl.min():
        swing_low.iloc[i]  = True
        sl_price.iloc[i]   = df['low'].iloc[i]

df['swing_high'] = swing_high
df['swing_low']  = swing_low
df['sh_price']   = sh_price
df['sl_price']   = sl_price

# BOS/CHoCH
bos_bull   = pd.Series(False, index=df.index)
bos_bear   = pd.Series(False, index=df.index)
choch_bull = pd.Series(False, index=df.index)
choch_bear = pd.Series(False, index=df.index)
last_sh    = None
last_sl    = None
trend      = None

for i in range(1, n):
    if df['swing_high'].iloc[i]:
        last_sh = df['sh_price'].iloc[i]
    if df['swing_low'].iloc[i]:
        last_sl = df['sl_price'].iloc[i]
    if last_sh is None or last_sl is None:
        continue
    close = df['close'].iloc[i]
    if close > last_sh:
        if trend == 'bull':
            bos_bull.iloc[i]   = True
        else:
            choch_bull.iloc[i] = True
        trend   = 'bull'
        last_sh = None
    elif close < last_sl:
        if trend == 'bear':
            bos_bear.iloc[i]   = True
        else:
            choch_bear.iloc[i] = True
        trend   = 'bear'
        last_sl = None

df['bos_bull']   = bos_bull
df['bos_bear']   = bos_bear
df['choch_bull'] = choch_bull
df['choch_bear'] = choch_bear

# Order Blocks
bull_ob_high = pd.Series(np.nan, index=df.index)
bull_ob_low  = pd.Series(np.nan, index=df.index)
bear_ob_high = pd.Series(np.nan, index=df.index)
bear_ob_low  = pd.Series(np.nan, index=df.index)

for i in range(5, n):
    if df['bos_bull'].iloc[i] or \
       df['choch_bull'].iloc[i]:
        for j in range(i-1, max(i-6, 0), -1):
            if df['close'].iloc[j] < df['open'].iloc[j]:
                bull_ob_high.iloc[i] = df['high'].iloc[j]
                bull_ob_low.iloc[i]  = df['low'].iloc[j]
                break
    if df['bos_bear'].iloc[i] or \
       df['choch_bear'].iloc[i]:
        for j in range(i-1, max(i-6, 0), -1):
            if df['close'].iloc[j] > df['open'].iloc[j]:
                bear_ob_high.iloc[i] = df['high'].iloc[j]
                bear_ob_low.iloc[i]  = df['low'].iloc[j]
                break

df['bull_ob_high'] = bull_ob_high
df['bull_ob_low']  = bull_ob_low
df['bear_ob_high'] = bear_ob_high
df['bear_ob_low']  = bear_ob_low

print(f"SMC components ready")
print(f"  Swing highs : {df['swing_high'].sum()}")
print(f"  Swing lows  : {df['swing_low'].sum()}")
print(f"  Bull BOS    : {df['bos_bull'].sum()}")
print(f"  Bear BOS    : {df['bos_bear'].sum()}")
print(f"  Bull CHoCH  : {df['choch_bull'].sum()}")
print(f"  Bear CHoCH  : {df['choch_bear'].sum()}")
print(f"  Bull OBs    : {df['bull_ob_high'].notna().sum()}")
print(f"  Bear OBs    : {df['bear_ob_high'].notna().sum()}")

Calculating SMC components...
SMC components ready
  Swing highs : 2608
  Swing lows  : 2959
  Bull BOS    : 226
  Bear BOS    : 329
  Bull CHoCH  : 307
  Bear CHoCH  : 306
  Bull OBs    : 504
  Bear OBs    : 619


In [14]:
print("Finding Powell anchor bars...")
trading_dates = pd.Series(df.index.date).unique()
anchors = []

for date in trading_dates:
    dt         = datetime(date.year, date.month, date.day)
    anchor_utc = get_anchor_hour_utc(dt)
    cutoff_utc = get_cutoff_hour_utc(dt)

    day_bars = df[df.index.date == date]

    # Find 10AM anchor bar
    anchor_bars = day_bars[
        (day_bars.index.hour   == anchor_utc) &
        (day_bars.index.minute == 0)
    ]
    if len(anchor_bars) == 0:
        continue

    anchor_price = anchor_bars.iloc[0]['open']
    anchor_time  = anchor_bars.index[0]

    # Window: 10AM to cutoff
    window_bars = day_bars[
        (day_bars.index >= anchor_time) &
        (day_bars.index.hour < cutoff_utc)
    ]
    if len(window_bars) < 5:
        continue

    anchors.append({
        'date'        : date,
        'anchor_time' : anchor_time,
        'anchor_price': anchor_price,
        'anchor_utc'  : anchor_utc,
        'cutoff_utc'  : cutoff_utc,
        'window_bars' : window_bars,
        'day_bars'    : day_bars
    })

print(f"Anchor bars found: {len(anchors)}")

# ── COMBINED SIGNAL DETECTION ─────────────────────────────
def detect_combined_signal(anchor, df,
                            min_manip_pct=0.001):
    """
    Combined Powell + SMC + LIP-SIM signal.

    LONG conditions:
    - Powell: price sweeps below anchor > 0.1%
              then closes back above
    - LIP-SIM: bull_trend, rsi_bull, macd_bull,
               has_trend at entry bar
    - SMC: recent bull BOS/CHoCH within 20 bars
           price in bull OB zone

    SHORT conditions:
    - Powell: price sweeps above anchor > 0.1%
              then closes back below
    - LIP-SIM: bear_trend, rsi_bear, macd_bear,
               has_trend at entry bar
    - SMC: recent bear BOS/CHoCH within 20 bars
           price in bear OB zone
    """
    anchor_price = anchor['anchor_price']
    anchor_time  = anchor['anchor_time']
    window_bars  = anchor['window_bars']

    # Step 1 — Manipulation window (first 6 bars = 30min)
    manip_bars = window_bars.iloc[:6]
    if len(manip_bars) < 3:
        return None

    manip_high = manip_bars['high'].max()
    manip_low  = manip_bars['low'].min()

    swept_above = manip_high > anchor_price
    swept_below = manip_low  < anchor_price

    if not (swept_above or swept_below):
        return None

    # Determine direction
    if swept_above and swept_below:
        up_sweep   = manip_high - anchor_price
        down_sweep = anchor_price - manip_low
        direction     = 'SHORT' if up_sweep >= down_sweep \
                        else 'LONG'
        manip_extreme = manip_high if direction == 'SHORT' \
                        else manip_low
    elif swept_above:
        direction     = 'SHORT'
        manip_extreme = manip_high
    else:
        direction     = 'LONG'
        manip_extreme = manip_low

    # Check minimum manipulation size
    manip_pct = abs(manip_extreme - anchor_price) / \
                anchor_price
    if manip_pct < min_manip_pct:
        return None

    # Step 2 — Look for reversal back through anchor
    reversal_bars = window_bars.iloc[3:]
    entry_bar     = None
    entry_price   = None
    entry_idx     = None

    for idx, bar in reversal_bars.iterrows():
        if direction == 'SHORT':
            if bar['close'] < anchor_price:
                entry_bar   = idx
                entry_price = bar['close']
                break
        else:
            if bar['close'] > anchor_price:
                entry_bar   = idx
                entry_price = bar['close']
                break

    if entry_price is None:
        return None

    # Step 3 — LIP-SIM confirmation at entry bar
    entry_row = df.loc[entry_bar]

    if direction == 'SHORT':
        lipsim_ok = (entry_row['bear_trend'] and
                     entry_row['rsi_bear']   and
                     entry_row['macd_bear']  and
                     entry_row['has_trend'])
    else:
        lipsim_ok = (entry_row['bull_trend'] and
                     entry_row['rsi_bull']   and
                     entry_row['macd_bull']  and
                     entry_row['has_trend'])

    if not lipsim_ok:
        return None

    # Step 4 — SMC confirmation
    # Look back 20 bars for recent BOS/CHoCH
    entry_pos = df.index.get_loc(entry_bar)
    lookback_start = max(0, entry_pos - 20)
    recent = df.iloc[lookback_start:entry_pos+1]

    if direction == 'SHORT':
        recent_smc = (recent['bos_bear'].any() or
                      recent['choch_bear'].any())
        # Check if price in bear OB zone
        active_ob = recent['bear_ob_high'].dropna()
        if len(active_ob) > 0:
            last_ob_high = recent['bear_ob_high']\
                           .dropna().iloc[-1]
            last_ob_low  = recent['bear_ob_low']\
                           .dropna().iloc[-1]
            in_ob = (entry_price <= last_ob_high and
                     entry_price >= last_ob_low)
        else:
            in_ob = False
    else:
        recent_smc = (recent['bos_bull'].any() or
                      recent['choch_bull'].any())
        active_ob = recent['bull_ob_high'].dropna()
        if len(active_ob) > 0:
            last_ob_high = recent['bull_ob_high']\
                           .dropna().iloc[-1]
            last_ob_low  = recent['bull_ob_low']\
                           .dropna().iloc[-1]
            in_ob = (entry_price <= last_ob_high and
                     entry_price >= last_ob_low)
        else:
            in_ob = False

    # Require either BOS/CHoCH OR OB — not both
    # (too strict requiring both on 5min)
    smc_ok = recent_smc or in_ob
    if not smc_ok:
        return None

    # Step 5 — Calculate levels
    risk = abs(entry_price - manip_extreme)
    if risk == 0:
        return None

    sl   = manip_extreme
    tp3  = entry_price - 3*risk if direction=='SHORT' \
           else entry_price + 3*risk
    tp5  = entry_price - 5*risk if direction=='SHORT' \
           else entry_price + 5*risk

    return {
        'date'         : anchor['date'],
        'direction'    : direction,
        'anchor_price' : anchor_price,
        'anchor_time'  : anchor_time,
        'manip_extreme': manip_extreme,
        'manip_pct'    : manip_pct,
        'entry_price'  : entry_price,
        'entry_bar'    : entry_bar,
        'sl'           : sl,
        'tp3'          : tp3,
        'tp5'          : tp5,
        'risk'         : risk,
        'lipsim_ok'    : lipsim_ok,
        'smc_ok'       : smc_ok,
        'in_ob'        : in_ob,
        'recent_smc'   : recent_smc,
        'day_bars'     : anchor['day_bars']
    }

# Detect all combined signals
signals = []
for anchor in anchors:
    signal = detect_combined_signal(
        anchor, df, min_manip_pct=0.001)
    if signal:
        signals.append(signal)

longs  = sum(1 for s in signals if s['direction']=='LONG')
shorts = sum(1 for s in signals if s['direction']=='SHORT')

print(f"\nCombined signals detected: {len(signals)}")
print(f"  LONG  : {longs}")
print(f"  SHORT : {shorts}")
print(f"  Rate  : {len(signals)/len(anchors):.0%} of days")

Finding Powell anchor bars...
Anchor bars found: 139

Combined signals detected: 4
  LONG  : 2
  SHORT : 2
  Rate  : 3% of days


In [15]:
def detect_combined_v2(anchor, df,
                        min_manip_pct=0.001):
    """
    Relaxed combined signal:
    - Powell: manipulation > 0.1% + reversal
    - LIP-SIM: trend + RSI + MACD + ADX aligned
    - SMC: recent BOS or CHoCH only (drop OB requirement)
    """
    anchor_price = anchor['anchor_price']
    anchor_time  = anchor['anchor_time']
    window_bars  = anchor['window_bars']

    # Manipulation
    manip_bars = window_bars.iloc[:6]
    if len(manip_bars) < 3:
        return None

    manip_high = manip_bars['high'].max()
    manip_low  = manip_bars['low'].min()
    swept_above = manip_high > anchor_price
    swept_below = manip_low  < anchor_price

    if not (swept_above or swept_below):
        return None

    if swept_above and swept_below:
        up_sweep      = manip_high - anchor_price
        down_sweep    = anchor_price - manip_low
        direction     = 'SHORT' if up_sweep >= down_sweep \
                        else 'LONG'
        manip_extreme = manip_high if direction=='SHORT' \
                        else manip_low
    elif swept_above:
        direction     = 'SHORT'
        manip_extreme = manip_high
    else:
        direction     = 'LONG'
        manip_extreme = manip_low

    manip_pct = abs(manip_extreme - anchor_price) / \
                anchor_price
    if manip_pct < min_manip_pct:
        return None

    # Reversal back through anchor
    reversal_bars = window_bars.iloc[3:]
    entry_bar     = None
    entry_price   = None

    for idx, bar in reversal_bars.iterrows():
        if direction == 'SHORT':
            if bar['close'] < anchor_price:
                entry_bar   = idx
                entry_price = bar['close']
                break
        else:
            if bar['close'] > anchor_price:
                entry_bar   = idx
                entry_price = bar['close']
                break

    if entry_price is None:
        return None

    # LIP-SIM confirmation
    entry_row = df.loc[entry_bar]
    if direction == 'SHORT':
        lipsim_ok = (entry_row['bear_trend'] and
                     entry_row['rsi_bear']   and
                     entry_row['macd_bear']  and
                     entry_row['has_trend'])
    else:
        lipsim_ok = (entry_row['bull_trend'] and
                     entry_row['rsi_bull']   and
                     entry_row['macd_bull']  and
                     entry_row['has_trend'])

    if not lipsim_ok:
        return None

    # SMC — BOS or CHoCH within 30 bars
    entry_pos = df.index.get_loc(entry_bar)
    lookback_start = max(0, entry_pos - 30)
    recent = df.iloc[lookback_start:entry_pos+1]

    if direction == 'SHORT':
        smc_ok = (recent['bos_bear'].any() or
                  recent['choch_bear'].any())
    else:
        smc_ok = (recent['bos_bull'].any() or
                  recent['choch_bull'].any())

    if not smc_ok:
        return None

    risk = abs(entry_price - manip_extreme)
    if risk == 0:
        return None

    sl  = manip_extreme
    tp3 = entry_price - 3*risk if direction=='SHORT' \
          else entry_price + 3*risk
    tp5 = entry_price - 5*risk if direction=='SHORT' \
          else entry_price + 5*risk

    return {
        'date'         : anchor['date'],
        'direction'    : direction,
        'anchor_price' : anchor_price,
        'manip_extreme': manip_extreme,
        'manip_pct'    : manip_pct,
        'entry_price'  : entry_price,
        'entry_bar'    : entry_bar,
        'sl'           : sl,
        'tp3'          : tp3,
        'tp5'          : tp5,
        'risk'         : risk,
        'day_bars'     : anchor['day_bars']
    }

# Test with different manip thresholds
print(f"{'='*55}")
print(f"  SIGNAL COUNT BY FILTER STRENGTH")
print(f"{'='*55}")
print(f"  {'Filter':<30} {'Signals':>8} {'Rate':>6}")
print(f"  {'-'*50}")

for min_m in [0.0005, 0.001, 0.002, 0.003]:
    sigs = []
    for anchor in anchors:
        s = detect_combined_v2(anchor, df, min_m)
        if s:
            sigs.append(s)
    rate = len(sigs)/len(anchors)
    l = sum(1 for s in sigs if s['direction']=='LONG')
    sh = sum(1 for s in sigs if s['direction']=='SHORT')
    print(f"  Manip>{min_m*100:.2f}% + LIP+SMC    "
          f"{len(sigs):>8}   {rate:>5.0%}  "
          f"(L:{l} S:{sh})")

print(f"{'='*55}")

# Use 0.001 as final threshold
signals_v2 = []
for anchor in anchors:
    s = detect_combined_v2(anchor, df, 0.001)
    if s:
        signals_v2.append(s)

print(f"\nFinal signals (Manip>0.1%): {len(signals_v2)}")

  SIGNAL COUNT BY FILTER STRENGTH
  Filter                          Signals   Rate
  --------------------------------------------------
  Manip>0.05% + LIP+SMC           7      5%  (L:2 S:5)
  Manip>0.10% + LIP+SMC           5      4%  (L:2 S:3)
  Manip>0.20% + LIP+SMC           3      2%  (L:2 S:1)
  Manip>0.30% + LIP+SMC           0      0%  (L:0 S:0)

Final signals (Manip>0.1%): 5


In [16]:
def fetch_h1(symbol="XAU/USD", outputsize=5000):
    url    = "https://api.twelvedata.com/time_series"
    params = {
        "symbol"    : symbol,
        "interval"  : "1h",
        "outputsize": outputsize,
        "apikey"    : API_KEY,
        "format"    : "JSON"
    }
    r    = requests.get(url, params=params)
    data = r.json()
    if data.get("status") == "error":
        print(f"Error: {data.get('message')}")
        return None
    df = pd.DataFrame(data["values"])
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df.set_index("datetime").sort_index()
    df = df.astype(float)
    df.columns = [c.lower() for c in df.columns]
    return df

print("Fetching H1 XAU/USD...")
time.sleep(8)
h1 = fetch_h1("XAU/USD", 5000)

if h1 is not None:
    print(f"H1 bars  : {len(h1)}")
    print(f"Range    : {h1.index[0].date()} "
          f"to {h1.index[-1].date()}")

    # Calculate H1 LIP-SIM indicators
    h1['ema_fast'] = ema(h1['close'], EMA_FAST)
    h1['ema_mid']  = ema(h1['close'], EMA_MID)
    h1['ema_slow'] = ema(h1['close'], EMA_SLOW)
    h1['rsi']      = rsi_calc(h1['close'], RSI_LEN)
    h1['macd_l'], h1['macd_s'] = macd_calc(h1['close'])
    h1['adx']      = adx_calc(h1['high'], h1['low'],
                               h1['close'], ADX_LEN)

    # H1 bias conditions
    h1['bear_bias'] = (
        (h1['ema_fast'] < h1['ema_mid']) &
        (h1['ema_fast'] < h1['ema_slow']) &
        (h1['rsi'] < 50) & (h1['rsi'] > 20) &
        (h1['macd_l'] < h1['macd_s']) &
        (h1['adx'] >= ADX_MIN)
    )
    h1['bull_bias'] = (
        (h1['ema_fast'] > h1['ema_mid']) &
        (h1['ema_fast'] > h1['ema_slow']) &
        (h1['rsi'] > 50) & (h1['rsi'] < 80) &
        (h1['macd_l'] > h1['macd_s']) &
        (h1['adx'] >= ADX_MIN)
    )

    bear_h = h1['bear_bias'].sum()
    bull_h = h1['bull_bias'].sum()
    print(f"\nH1 LIP-SIM bias:")
    print(f"  Bear bias hours: {bear_h}")
    print(f"  Bull bias hours: {bull_h}")
    print(f"  Neutral hours  : "
          f"{len(h1)-bear_h-bull_h}")

Fetching H1 XAU/USD...
H1 bars  : 5000
Range    : 2025-12-19 to 2026-07-19

H1 LIP-SIM bias:
  Bear bias hours: 685
  Bull bias hours: 543
  Neutral hours  : 3772


In [17]:
def get_h1_bias(entry_time, h1_df):
    """
    Get H1 LIP-SIM bias at the time of 5min entry.
    Look at the current H1 bar containing entry_time.
    """
    # Find the H1 bar that contains this 5min timestamp
    h1_times = h1_df.index[h1_df.index <= entry_time]
    if len(h1_times) == 0:
        return 'neutral'
    h1_bar = h1_df.loc[h1_times[-1]]
    if h1_bar['bear_bias']:
        return 'bear'
    elif h1_bar['bull_bias']:
        return 'bull'
    else:
        return 'neutral'

def detect_mtf_signal(anchor, df, h1_df,
                       min_manip_pct=0.001):
    """
    Multi-timeframe combined signal:
    H1 LIP-SIM bias + 5min Powell + 5min SMC
    """
    anchor_price = anchor['anchor_price']
    anchor_time  = anchor['anchor_time']
    window_bars  = anchor['window_bars']

    # Manipulation
    manip_bars  = window_bars.iloc[:6]
    if len(manip_bars) < 3:
        return None

    manip_high  = manip_bars['high'].max()
    manip_low   = manip_bars['low'].min()
    swept_above = manip_high > anchor_price
    swept_below = manip_low  < anchor_price

    if not (swept_above or swept_below):
        return None

    if swept_above and swept_below:
        up_sweep      = manip_high - anchor_price
        down_sweep    = anchor_price - manip_low
        direction     = 'SHORT' if up_sweep >= down_sweep \
                        else 'LONG'
        manip_extreme = manip_high if direction=='SHORT' \
                        else manip_low
    elif swept_above:
        direction     = 'SHORT'
        manip_extreme = manip_high
    else:
        direction     = 'LONG'
        manip_extreme = manip_low

    manip_pct = abs(manip_extreme - anchor_price) / \
                anchor_price
    if manip_pct < min_manip_pct:
        return None

    # Reversal back through anchor
    reversal_bars = window_bars.iloc[3:]
    entry_bar     = None
    entry_price   = None

    for idx, bar in reversal_bars.iterrows():
        if direction == 'SHORT':
            if bar['close'] < anchor_price:
                entry_bar   = idx
                entry_price = bar['close']
                break
        else:
            if bar['close'] > anchor_price:
                entry_bar   = idx
                entry_price = bar['close']
                break

    if entry_price is None:
        return None

    # H1 LIP-SIM bias check
    h1_bias = get_h1_bias(entry_bar, h1_df)
    if direction == 'SHORT' and h1_bias != 'bear':
        return None
    if direction == 'LONG'  and h1_bias != 'bull':
        return None

    # 5min SMC confirmation
    entry_pos      = df.index.get_loc(entry_bar)
    lookback_start = max(0, entry_pos - 30)
    recent         = df.iloc[lookback_start:entry_pos+1]

    if direction == 'SHORT':
        smc_ok = (recent['bos_bear'].any() or
                  recent['choch_bear'].any())
    else:
        smc_ok = (recent['bos_bull'].any() or
                  recent['choch_bull'].any())

    if not smc_ok:
        return None

    risk = abs(entry_price - manip_extreme)
    if risk == 0:
        return None

    sl  = manip_extreme
    tp3 = entry_price - 3*risk if direction=='SHORT' \
          else entry_price + 3*risk
    tp5 = entry_price - 5*risk if direction=='SHORT' \
          else entry_price + 5*risk

    return {
        'date'         : anchor['date'],
        'direction'    : direction,
        'anchor_price' : anchor_price,
        'manip_extreme': manip_extreme,
        'manip_pct'    : round(manip_pct*100, 3),
        'entry_price'  : entry_price,
        'entry_bar'    : entry_bar,
        'sl'           : sl,
        'tp3'          : tp3,
        'tp5'          : tp5,
        'risk'         : risk,
        'h1_bias'      : h1_bias,
        'day_bars'     : anchor['day_bars']
    }

# Test signal counts at different thresholds
print(f"{'='*58}")
print(f"  MTF SIGNAL COUNT (H1 LIP-SIM + 5min Powell+SMC)")
print(f"{'='*58}")
print(f"  {'Filter':<25} {'N':>5} {'Rate':>6} "
      f"{'Long':>6} {'Short':>6}")
print(f"  {'-'*53}")

for min_m in [0.0005, 0.001, 0.002, 0.003]:
    sigs = []
    for anchor in anchors:
        s = detect_mtf_signal(anchor, df, h1, min_m)
        if s:
            sigs.append(s)
    l  = sum(1 for s in sigs if s['direction']=='LONG')
    sh = sum(1 for s in sigs if s['direction']=='SHORT')
    print(f"  Manip>{min_m*100:.2f}%            "
          f"{len(sigs):>5}  "
          f"{len(sigs)/len(anchors):>5.0%}  "
          f"{l:>6}  {sh:>6}")

print(f"{'='*58}")

# Use 0.05% as final threshold
signals_mtf = []
for anchor in anchors:
    s = detect_mtf_signal(anchor, df, h1, 0.0005)
    if s:
        signals_mtf.append(s)

print(f"\nFinal MTF signals: {len(signals_mtf)}")
longs  = sum(1 for s in signals_mtf
             if s['direction']=='LONG')
shorts = sum(1 for s in signals_mtf
             if s['direction']=='SHORT')
print(f"  LONG  : {longs}")
print(f"  SHORT : {shorts}")

  MTF SIGNAL COUNT (H1 LIP-SIM + 5min Powell+SMC)
  Filter                        N   Rate   Long  Short
  -----------------------------------------------------
  Manip>0.05%                6     4%       0       6
  Manip>0.10%                5     4%       0       5
  Manip>0.20%                1     1%       0       1
  Manip>0.30%                0     0%       0       0

Final MTF signals: 6
  LONG  : 0
  SHORT : 6


In [18]:
# Diagnose why signals are so rare
print("SIGNAL DROPOUT ANALYSIS")
print("=" * 55)

powell_ok    = 0
manip_ok     = 0
reversal_ok  = 0
h1_bias_ok   = 0
smc_ok_count = 0

for anchor in anchors:
    anchor_price = anchor['anchor_price']
    window_bars  = anchor['window_bars']

    # Check manipulation
    manip_bars  = window_bars.iloc[:6]
    if len(manip_bars) < 3:
        continue
    powell_ok += 1

    manip_high  = manip_bars['high'].max()
    manip_low   = manip_bars['low'].min()
    swept_above = manip_high > anchor_price
    swept_below = manip_low  < anchor_price
    if not (swept_above or swept_below):
        continue

    manip_pct = max(
        abs(manip_high - anchor_price),
        abs(anchor_price - manip_low)
    ) / anchor_price
    if manip_pct < 0.0005:
        continue
    manip_ok += 1

    # Check reversal
    if swept_above and swept_below:
        up   = manip_high - anchor_price
        down = anchor_price - manip_low
        direction     = 'SHORT' if up >= down else 'LONG'
        manip_extreme = manip_high if direction=='SHORT' \
                        else manip_low
    elif swept_above:
        direction = 'SHORT'; manip_extreme = manip_high
    else:
        direction = 'LONG';  manip_extreme = manip_low

    reversal_bars = window_bars.iloc[3:]
    entry_bar = None; entry_price = None
    for idx, bar in reversal_bars.iterrows():
        if direction=='SHORT' and \
           bar['close'] < anchor_price:
            entry_bar = idx
            entry_price = bar['close']
            break
        elif direction=='LONG' and \
             bar['close'] > anchor_price:
            entry_bar = idx
            entry_price = bar['close']
            break

    if entry_price is None:
        continue
    reversal_ok += 1

    # Check H1 bias
    h1_bias = get_h1_bias(entry_bar, h1)
    if direction=='SHORT' and h1_bias != 'bear':
        continue
    if direction=='LONG'  and h1_bias != 'bull':
        continue
    h1_bias_ok += 1

    # Check SMC
    entry_pos = df.index.get_loc(entry_bar)
    recent    = df.iloc[max(0,entry_pos-30):entry_pos+1]
    if direction=='SHORT':
        smc = (recent['bos_bear'].any() or
               recent['choch_bear'].any())
    else:
        smc = (recent['bos_bull'].any() or
               recent['choch_bull'].any())
    if smc:
        smc_ok_count += 1

print(f"  Total anchor days    : {len(anchors)}")
print(f"  Enough manip bars    : {powell_ok}")
print(f"  Manip > 0.05%        : {manip_ok}")
print(f"  Reversal confirmed   : {reversal_ok}")
print(f"  H1 bias aligned      : {h1_bias_ok}  "
      f"← biggest dropout")
print(f"  SMC confirmed        : {smc_ok_count}")
print(f"{'='*55}")
print(f"\nH1 bias is filtering out "
      f"{reversal_ok - h1_bias_ok} signals")
print(f"({(reversal_ok-h1_bias_ok)/reversal_ok:.0%} "
      f"of reversals rejected by H1 bias)")

SIGNAL DROPOUT ANALYSIS
  Total anchor days    : 139
  Enough manip bars    : 139
  Manip > 0.05%        : 99
  Reversal confirmed   : 61
  H1 bias aligned      : 8  ← biggest dropout
  SMC confirmed        : 6

H1 bias is filtering out 53 signals
(87% of reversals rejected by H1 bias)


In [19]:
def get_h1_bias_simple(entry_time, h1_df):
    """
    Simplified H1 bias — EMA trend only.
    SHORT bias: H1 EMA fast < EMA slow
    LONG  bias: H1 EMA fast > EMA slow
    """
    h1_times = h1_df.index[h1_df.index <= entry_time]
    if len(h1_times) == 0:
        return 'neutral'
    h1_bar = h1_df.loc[h1_times[-1]]
    if h1_bar['ema_fast'] < h1_bar['ema_slow']:
        return 'bear'
    elif h1_bar['ema_fast'] > h1_bar['ema_slow']:
        return 'bull'
    else:
        return 'neutral'

def detect_mtf_simple(anchor, df, h1_df,
                       min_manip_pct=0.0005):
    """
    Simplified MTF signal:
    H1 EMA trend bias + 5min Powell + 5min SMC BOS/CHoCH
    """
    anchor_price = anchor['anchor_price']
    window_bars  = anchor['window_bars']

    # Manipulation
    manip_bars  = window_bars.iloc[:6]
    if len(manip_bars) < 3:
        return None

    manip_high  = manip_bars['high'].max()
    manip_low   = manip_bars['low'].min()
    swept_above = manip_high > anchor_price
    swept_below = manip_low  < anchor_price

    if not (swept_above or swept_below):
        return None

    if swept_above and swept_below:
        up   = manip_high - anchor_price
        down = anchor_price - manip_low
        direction     = 'SHORT' if up >= down else 'LONG'
        manip_extreme = manip_high if direction=='SHORT' \
                        else manip_low
    elif swept_above:
        direction = 'SHORT'; manip_extreme = manip_high
    else:
        direction = 'LONG';  manip_extreme = manip_low

    manip_pct = abs(manip_extreme - anchor_price) / \
                anchor_price
    if manip_pct < min_manip_pct:
        return None

    # Reversal
    reversal_bars = window_bars.iloc[3:]
    entry_bar     = None
    entry_price   = None

    for idx, bar in reversal_bars.iterrows():
        if direction=='SHORT' and \
           bar['close'] < anchor_price:
            entry_bar = idx; entry_price = bar['close']
            break
        elif direction=='LONG' and \
             bar['close'] > anchor_price:
            entry_bar = idx; entry_price = bar['close']
            break

    if entry_price is None:
        return None

    # H1 EMA trend bias only
    h1_bias = get_h1_bias_simple(entry_bar, h1_df)
    if direction=='SHORT' and h1_bias != 'bear':
        return None
    if direction=='LONG'  and h1_bias != 'bull':
        return None

    # 5min SMC BOS/CHoCH
    entry_pos = df.index.get_loc(entry_bar)
    recent    = df.iloc[max(0,entry_pos-30):entry_pos+1]

    if direction=='SHORT':
        smc_ok = (recent['bos_bear'].any() or
                  recent['choch_bear'].any())
    else:
        smc_ok = (recent['bos_bull'].any() or
                  recent['choch_bull'].any())

    if not smc_ok:
        return None

    risk = abs(entry_price - manip_extreme)
    if risk == 0:
        return None

    sl  = manip_extreme
    tp3 = entry_price - 3*risk if direction=='SHORT' \
          else entry_price + 3*risk
    tp5 = entry_price - 5*risk if direction=='SHORT' \
          else entry_price + 5*risk

    return {
        'date'         : anchor['date'],
        'direction'    : direction,
        'anchor_price' : anchor_price,
        'manip_extreme': manip_extreme,
        'manip_pct'    : round(manip_pct*100, 3),
        'entry_price'  : entry_price,
        'entry_bar'    : entry_bar,
        'sl'           : sl,
        'tp3'          : tp3,
        'tp5'          : tp5,
        'risk'         : risk,
        'h1_bias'      : h1_bias,
        'day_bars'     : anchor['day_bars']
    }

# Test signal counts
print(f"{'='*58}")
print(f"  MTF SIMPLIFIED — H1 EMA trend + Powell + SMC")
print(f"{'='*58}")
print(f"  {'Filter':<25} {'N':>5} {'Rate':>6} "
      f"{'Long':>6} {'Short':>6}")
print(f"  {'-'*53}")

for min_m in [0.0005, 0.001, 0.002, 0.003]:
    sigs = []
    for anchor in anchors:
        s = detect_mtf_simple(anchor, df, h1, min_m)
        if s:
            sigs.append(s)
    l  = sum(1 for s in sigs if s['direction']=='LONG')
    sh = sum(1 for s in sigs if s['direction']=='SHORT')
    print(f"  Manip>{min_m*100:.2f}%            "
          f"{len(sigs):>5}  "
          f"{len(sigs)/len(anchors):>5.0%}  "
          f"{l:>6}  {sh:>6}")

print(f"{'='*58}")

  MTF SIMPLIFIED — H1 EMA trend + Powell + SMC
  Filter                        N   Rate   Long  Short
  -----------------------------------------------------
  Manip>0.05%               10     7%       1       9
  Manip>0.10%                7     5%       1       6
  Manip>0.20%                1     1%       0       1
  Manip>0.30%                0     0%       0       0
